## Module 1: Agentic RAG

This module consists of building a simple RAG pipeline using keyword search, and making it agentic, so the LLM decides when and what to search instead of running a fixed pipeline.

### Part 1: RAG

RAG (*Retrieval-Augmented Generation*) is a technique used to improve the accuracy and reliability of LLMs (*Large Language Models*) by fetching facts from an external knowledge base before generating a response. In such cases, where an external knowledge base is the context, usual LLMs can only give generic answers without having access to that specific knowledge. 

- **Why RAG:**

In [1]:
# make sure .env file is loaded
from dotenv import load_dotenv
print(load_dotenv())

# check whether the OpenAI client works
from openai import OpenAI
openai_client = OpenAI()

True


In [2]:
# define a function to talk to llm and test it
def llm(prompt):
    response = openai_client.responses.create(
        model = 'gpt-5.4-mini',
        input = prompt
    )
    return response.output_text

llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s on your mind?'

Here we are asking an external knowledge base specific question without providing th context:

In [3]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes—usually you can still join, but it depends on the course policy and whether enrollment is still open.

If you want, I can help you draft a quick message like:

> Hi, I just discovered the course and am very interested in joining. Is it still possible to enroll at this stage? If so, please let me know the next steps.

If you tell me the course name or context, I can make it more specific and polite.


The model returns a very generic answer since it has no knowledge about that specific course. Here we provide the model some context to depend on. The prompt doesn't end with "Answer:". With older models like GPT-3 it was added to nudge the model into completing the sentence. Modern models don't need the hint.

In [4]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.


- **Getting Data:**

Generation in RAG is the LLM producing the text, and retrieval is the search. Relevant documents are retrieved from the knowledge base and used to augment what the LLM generates. That search step is what gives the LLM the context it needs to answer correctly. What was done above was naive, since FAQ entry already held the answer and it was pasted as context by hand. What the actual RAG does instead is to perform search automatically. Then LLM sees only the documents handed to it, so the model is only as good as the retrieved data.

In [6]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

The goal is to create a RAG using such a function as above which would get a user question, search it within the knowledge base, build the prompt including the question and filtered search results, than forwarding it to an LLM to get a response. 

In [7]:
import requests

# get courses data from the faq page 
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

# get q&a pairs for each course and add to documents
for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [8]:
# check the courses
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [9]:
# check a sample q&a
documents[600]

{'id': 'eae0fb50aa',
 'course': 'llm-zoomcamp',
 'section': 'Capstone Project',
 'question': 'When and how will we be assigned projects for review/grading?',
 'answer': 'After the submission deadline has passed, an Excel sheet will be shared with 3 projects being assigned to each participant.'}

Usually data is not prepared into json files, and requires to be parsed in order to be provided as a knowledge base.

In this RAG pipeline, Q&A dataset is the knowledge base. All documents from the knowledge base are indexed. When a question is asked, the index is searched and the search returns the most relevant entries. The found entries are given to the LLM as context and the LLM generates an answer based on the context.

- **Search:**

The `question` and `answer` fields contain the text to search through. The `course` field is used to filter by course. The `section` field helps with ranking - knowing which part of the course a question belongs to is useful context.

Since there are many documents, to index them, search matters. Rather than vector/semantic search, this RAG will use text/lexical search. Instead of search libraries like Apache Lucene, Elasticsearch, Solr, and others, the lightweight minsearch from the course is used.

In [10]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

Text fields are the fields to search through. The search engine looks at these fields and tokenizes them, when a query is entered. It splits text into words, lowercases them, removes stop words, and ranks the results by relevance.

Keyword fields are for exact matching. The results must come from the specified course, no matter what ranking or boosting you do for text fields.

In [11]:
# try a search using a question
question = "I just discovered the course. Can I join now?"

# to get all related answers from all the courses
# index.search(question)

# to get filtered results from a selected course with boosting specific fields
search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [12]:
# wrap searching in a function
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [13]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

- **Prompt:**

A prompt usually consists of two parts; instructions (system prompt) and user prompt. Instructions tell LLM to how to behave and they are the same for every request. User prompt changes with every request and carries the actual question as well as the retrieved content. The prompt is the bridge between the search and the LLM.

In [14]:
# instructions remain same
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

# user prompt changes, hence we use a template
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

To provide the context to the user prompt, build the context from the search results by making it easy for LLM to read.

In [15]:
# all questions from the search results
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [16]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [17]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [18]:
# combine the question with the context into user prompt
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [19]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

- **LLM:**

In [21]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

# return the full output to observe
print(response.model_dump_json(indent=2))

{
  "id": "resp_0a7267ae30fdc971006a3039449c58819d828da846984191ec",
  "created_at": 1781545284.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_0a7267ae30fdc971006a3039454210819dbc7634ea741e9e31",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can still join now.\n\nIf you want a certificate, make sure to submit your project while the course is still accepting submissions.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1781545285.0,
  "conversation": null,
  "max_output_tokens": null,
  "max_tool_calls": nu

In [27]:
# since the access to the actual answer is long, a short form is used
response.output[0].content[0].text.strip() == response.output_text

True

In [28]:
response.usage

ResponseUsage(input_tokens=334, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=32, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=366)

For the selected gpt-5.4-mini model, the total cost so far can be calculated as follows. The input price and output price details are shared on the model page: [gpt-5.4-mini docs](https://developers.openai.com/api/docs/models/gpt-5.4-mini)

In [29]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0003945

Previously only one string was sent as input and a response was received for it. In practice a message history, which is a list of messages where each message has a role, is sent. ChatGPT conversation starts with a hidden system prompt that tells the LLM how to behave, which is invisible. After that, messages and the LLM's replies alternate. The LLM has no memory of its own, so it needs the full history passed in to continue the conversation. This exercise do not require a multi-turn chat, but the instructions will be separated from the user prompt by sending two messages:

- `developer` - system-level instructions (how the LLM should behave)
- `user` - the actual prompt with the question and context

In [30]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [31]:
# wrap the request into a function
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

- **Full RAG:**

As all components are completed, RAG can be used to answer user questions:

In [32]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [33]:
answer = rag(question)
print(answer)

Yes, you can still join now.

If you want a certificate, you need to submit your project while submissions are still open.


In [34]:
answer = rag("How do I get a certificate?")
print(answer)

You can get a certificate only if you finish the course with a live cohort. Self-paced mode does not qualify.

To receive the certificate, you also need to:
- pass the Capstone project
- peer-review 3 capstones during the course run

If you want the certificate to show your real name, make sure to update your official name in your course profile.
